# Day 2 — 5-Fold Cross-Validation

## Overview

This task focuses on evaluating a classification model using 5-fold cross-validation.

The goal is to estimate the model's performance across multiple train-validation splits instead of relying on a single validation split.

The Random Forest model from Week 3 is evaluated using stratified 5-fold cross-validation. The mean and standard deviation of the cross-validation scores are reported and compared with the single-split validation score obtained on Day 1.

## Importing the Required Libraries

The following libraries are imported for data loading, preprocessing, data splitting, building machine learning pipelines, and training classification models.


In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

## Loading the Dataset

The Stroke Prediction dataset is loaded using Pandas.

The dataset was previously explored and analyzed during Week 3, where exploratory data analysis (EDA) was performed to understand the dataset, examine feature distributions, identify missing values, and investigate the target variable.

In [2]:
df = pd.read_csv("healthcare-dataset-stroke-data.csv")
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## Defining Features and Target

The target variable is `stroke`, which represents whether a patient experienced a stroke.

The `id` column was removed because it is only an identifier and does not provide meaningful information for predicting stroke.

The remaining columns are used as input features `X`, while `stroke` is stored as the target variable `y`.


In [3]:
X = df.drop(columns=["stroke", "id"])
y = df["stroke"]

## Identifying Numerical and Categorical Features

The input features are divided into numerical and categorical features based on their data types.

Numerical features will be imputed and standardized, while categorical features will be imputed and one-hot encoded.

In [15]:
categorical_features = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status"
]

numerical_features = [
    "age",
    "hypertension",
    "heart_disease",
    "avg_glucose_level",
    "bmi"
]

## Building the Preprocessing Pipeline

A `ColumnTransformer` is used to apply different preprocessing steps to numerical and categorical features.

For numerical features:
- Missing values are replaced using the median.
- Features are standardized using `StandardScaler`.

For categorical features:
- Missing values are replaced using the most frequent value.
- Categorical variables are converted into numerical features using `OneHotEncoder`.

The preprocessing steps are included inside the machine learning pipeline so that they are fitted separately within each cross-validation fold. This prevents data leakage between the training and validation portions of each fold.

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numerical_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

## Building the Random Forest Pipeline

A Random Forest classifier from Week 3 is used for the classification task.

The preprocessing steps and Random Forest classifier are combined into a single pipeline.

Keeping preprocessing and the classifier in the same pipeline ensures that preprocessing is performed correctly inside each cross-validation fold.

In [17]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

## Setting Up Stratified 5-Fold Cross-Validation

Since this is a binary classification problem and the `stroke` classes are highly imbalanced, stratified folds are used.

`StratifiedKFold` preserves approximately the same class distribution in each fold.

The dataset is divided into five folds. In each iteration, four folds are used for training and one fold is used for validation.

In [ ]:
cv = StratifiedKFold( n_splits=5,
                      shuffle=True,
                      random_state=42
)


## Performing 5-Fold Cross-Validation

The `cross_val_score` function is used to evaluate the Random Forest pipeline across the five stratified folds.

Unlike the single train-validation split used on Day 1, cross-validation evaluates the model using multiple different validation folds.

In [26]:
scores = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)
print("Fold scores:", scores)


Fold scores: [0.95205479 0.9481409  0.95107632 0.95205479 0.94911937]


## Fold Scores

The 5-fold stratified cross-validation produced the following accuracy scores:

* **Fold 1:** 95.21%
* **Fold 2:** 94.81%
* **Fold 3:** 95.11%
* **Fold 4:** 95.21%
* **Fold 5:** 94.91%

The scores are relatively close to each other, indicating that the Random Forest model performs consistently across the five different validation folds.

The highest accuracy was achieved in **Fold 1 and Fold 4 (95.21%)**, while the lowest accuracy was achieved in **Fold 2 (94.81%)**.

## Mean and Standard Deviation

The mean of the five cross-validation scores represents the average model accuracy across all folds.

The standard deviation measures how much the model's performance varies between folds.

A smaller standard deviation indicates that the model's performance is more consistent across different subsets of the data.

In [27]:
mean_score = scores.mean()
std_score = scores.std()

print(f"Mean CV Accuracy: {mean_score:.4f}")
print(f"Standard Deviation: {std_score:.4f}")

Mean CV Accuracy: 0.9505
Standard Deviation: 0.0016


## Cross-Validation Results

The Random Forest model was evaluated using 5-fold stratified cross-validation.

The model achieved a **mean cross-validation accuracy of 95.05%** with a **standard deviation of 0.16%**.

The relatively small standard deviation indicates that the model's performance was highly consistent across the five validation folds, with only minor variation between them.

These results suggest that the model provides stable performance across different subsets of the dataset.


## Comparison with Day 1

On Day 1, the Random Forest model achieved a validation accuracy of **95.11%** using a single train-validation split.

In this task, the same model achieved a mean accuracy of **95.05%** using 5-fold stratified cross-validation.

Although the Day 1 validation score is slightly higher, this does not mean that it represents better model performance. The difference is only **0.06 percentage points** and can be attributed to the different data splits used for evaluation.

The cross-validation estimate is considered more reliable because it is based on the model's performance across five different validation folds rather than a single validation subset.

Therefore, the **95.05% cross-validation mean provides a more robust estimate of the model's generalization performance**, while the Day 1 score represents the model's performance on one specific validation split.


## Additional Verification: Class Distribution Across Folds

As an additional verification step, inspired by an external reference, the class distribution of the validation set in each fold was examined.

This check confirms that `StratifiedKFold` maintains a similar proportion of Stroke and No Stroke cases across all five folds.

The results show that each fold contains approximately **4.8% Stroke cases** and **95.2% No Stroke cases**, confirming that the class distribution is preserved consistently across the folds.

This additional check provides a practical verification of the effect of stratification and helps demonstrate why it is appropriate for this highly imbalanced classification problem.

In [28]:
for fold, (_, val_idx) in enumerate(cv.split(X, y), 1):
    fold_y = y.iloc[val_idx]
    
    print(
        f"Fold {fold}: "
        f"Stroke = {fold_y.mean():.4f}, "
        f"No Stroke = {1 - fold_y.mean():.4f}"
    )

Fold 1: Stroke = 0.0489, No Stroke = 0.9511
Fold 2: Stroke = 0.0489, No Stroke = 0.9511
Fold 3: Stroke = 0.0489, No Stroke = 0.9511
Fold 4: Stroke = 0.0489, No Stroke = 0.9511
Fold 5: Stroke = 0.0479, No Stroke = 0.9521


## Stratified Folds

Stratified 5-fold cross-validation was used by applying `StratifiedKFold` with `n_splits=5`.

Stratification is important for this classification task because the `stroke` target variable is highly imbalanced, with significantly fewer stroke cases than non-stroke cases.

`StratifiedKFold` preserves approximately the same proportion of each class in every fold. This ensures that each validation fold contains a representative distribution of both stroke and non-stroke cases.

Without stratification, some folds could contain an unbalanced or unrepresentative number of stroke cases, which could make the evaluation less reliable.
